# LESS — Phase 4
## Downstream LoRA fine-tuning: Random-450 vs LESS-450

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import csv
import json
import math
import random
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this experiment.")

DEVICE = torch.device("cuda:0")

print("GPU    :", torch.cuda.get_device_name(DEVICE))
print("VRAM   :", f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3:.2f} GiB")

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This Phase-1 configuration requires BF16 support.")

print("BF16 supported:", torch.cuda.is_bf16_supported())

torch.cuda.empty_cache()


Python : 3.12.13
PyTorch: 2.10.0+cu128
CUDA   : 12.8
GPU    : Tesla T4
VRAM   : 14.56 GiB
BF16 supported: True


In [2]:
subprocess.check_call([
    "pip", "install", "-q", "liger-kernel"
])

from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
)

from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss

print("Imports OK.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.3/644.3 kB 13.0 MB/s eta 0:00:00
Imports OK.


In [3]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m","pip","install","-q","--no-deps","torchao>=0.16.0"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.4 MB/s eta 0:00:00


0

In [4]:
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 2048
DTYPE = torch.bfloat16

LORA_R = 128
LORA_ALPHA = 512
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
MICRO_BATCH_SIZE = 1
EFFECTIVE_BATCH_SIZE = 128
GRAD_ACCUM_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

RUN_RANDOM_450 = True
RUN_LESS_450 = True

OUTPUT_ROOT = Path("./less_phase4_training")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ROOT = OUTPUT_ROOT / "experiments"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_ROOT = OUTPUT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"Model          : {MODEL_NAME}")
print(f"LoRA            : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"LR              : {LEARNING_RATE}")
print(f"Epochs          : {NUM_EPOCHS}")
print(f"Effective batch : {EFFECTIVE_BATCH_SIZE}")


Configuration loaded.
Model          : Qwen/Qwen2.5-1.5B
LoRA            : r=128, alpha=512, dropout=0.1
LR              : 2e-05
Epochs          : 4
Effective batch : 128


In [5]:
DATA_FILES = {
    "flan_v2": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/flan_v2_mini.jsonl"),
    "cot":      Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/cot_mini.jsonl"),
    "dolly":    Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/dolly_mini.jsonl"),
    "oasst1":   Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/oasst1_mini.jsonl"),
}

MMLU_ROOT = Path("/kaggle/input/datasets/tanmairaghava/mmlu-data/mmlu")
MMLU_DEV_DIR = MMLU_ROOT / "dev"
MMLU_TEST_DIR = MMLU_ROOT / "test"

LESS_SELECTED_PATH = None

CANDIDATE_COUNT = 9000
RANDOM_450_COUNT = 450

for name, p in DATA_FILES.items():
    print(f"{name:10s} exists:", p.exists(), p)

print("MMLU dev :", MMLU_DEV_DIR.exists(), MMLU_DEV_DIR)
print("MMLU test:", MMLU_TEST_DIR.exists(), MMLU_TEST_DIR)

flan_v2    exists: True /kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/flan_v2_mini.jsonl
cot        exists: True /kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/cot_mini.jsonl
dolly      exists: True /kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/dolly_mini.jsonl
oasst1     exists: True /kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/oasst1_mini.jsonl
MMLU dev : True /kaggle/input/datasets/tanmairaghava/mmlu-data/mmlu/dev
MMLU test: True /kaggle/input/datasets/tanmairaghava/mmlu-data/mmlu/test


In [6]:
source_datasets = {}

for name, path in DATA_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing candidate source: {path}")
    ds = load_dataset("json", data_files=str(path), split="train")
    source_datasets[name] = ds
    print(f"{name:10s}: {len(ds):,}")

candidate_dataset = concatenate_datasets(list(source_datasets.values()))

assert len(candidate_dataset) == CANDIDATE_COUNT, (
    f"Expected {CANDIDATE_COUNT} candidates, got {len(candidate_dataset)}."
)

print(f"\nCandidate pool: {len(candidate_dataset):,}")
print("Columns:", candidate_dataset.column_names)


Generating train split: 0 examples [00:00, ? examples/s]

flan_v2   : 3,000


Generating train split: 0 examples [00:00, ? examples/s]

cot       : 3,000


Generating train split: 0 examples [00:00, ? examples/s]

dolly     : 1,500


Generating train split: 0 examples [00:00, ? examples/s]

oasst1    : 1,500

Candidate pool: 9,000
Columns: ['dataset', 'id', 'messages']


In [7]:
def locate_less_selection():
    candidates = []

    if LESS_SELECTED_PATH is not None:
        p = Path(LESS_SELECTED_PATH)
        candidates.append(p)

    candidates.extend([
        Path("./less_selected_top450.jsonl"),
        Path("./less_selected_top450.json"),
        Path("/kaggle/working/less_selected_top450.jsonl"),
        Path("/kaggle/working/less_selected_top450.json"),
    ])

    root = Path("/kaggle/input")
    if root.exists():
        candidates.extend(root.rglob("less_selected_top450.jsonl"))
        candidates.extend(root.rglob("less_selected_top450.json"))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        "Could not locate less_selected_top450.json/jsonl. "
        "Set LESS_SELECTED_PATH in Cell 4 to the mounted file path."
    )

less_path = locate_less_selection()
print("LESS selection:", less_path)

def load_selection_records(path: Path):
    if path.suffix.lower() == ".jsonl":
        with open(path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]

    if not isinstance(obj, list):
        raise ValueError("Expected a JSON list or JSONL file.")

    return obj

less_records = load_selection_records(less_path)

assert len(less_records) == 450, (
    f"Expected 450 LESS-selected records, got {len(less_records)}."
)

less_dataset = None

if all("messages" in r for r in less_records):
    less_dataset = load_dataset(
        "json",
        data_files=str(less_path),
        split="train",
    )

print(f"LESS examples loaded: {len(less_records)}")
print("First record keys:", list(less_records[0].keys()))


LESS selection: /kaggle/input/notebooks/manasaindusrikarri/less-phase3-mmlu/phase3_selection/less_selected_top450.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

LESS examples loaded: 450
First record keys: ['dataset', 'id', 'messages', '_candidate_index', '_rank', '_score', '_best_mmlu_subject']


In [8]:
rng = np.random.default_rng(SEED)

random_indices = rng.choice(
    CANDIDATE_COUNT,
    size=RANDOM_450_COUNT,
    replace=False,
)

random_indices = np.sort(random_indices)

random_dataset = candidate_dataset.select(random_indices.tolist())

assert len(random_dataset) == 450

random_index_path = RESULTS_ROOT / "random450_seed42_indices.json"
with open(random_index_path, "w", encoding="utf-8") as f:
    json.dump(random_indices.tolist(), f, indent=2)

print("Random-450 built.")
print("Seed:", SEED)
print("Saved indices:", random_index_path)
print("Overlap with LESS will be checked below.")


Random-450 built.
Seed: 42
Saved indices: less_phase4_training/results/random450_seed42_indices.json
Overlap with LESS will be checked below.


In [9]:
# Phase 3 stored _candidate_index for the selected records.
less_candidate_indices = [
    int(r["_candidate_index"])
    for r in less_records
    if "_candidate_index" in r
]

if len(less_candidate_indices) == 450:
    assert len(set(less_candidate_indices)) == 450
    assert all(0 <= i < CANDIDATE_COUNT for i in less_candidate_indices)

    overlap = len(set(less_candidate_indices) & set(random_indices.tolist()))

    print("LESS candidate-index metadata found.")
    print("LESS size   :", len(less_candidate_indices))
    print("Random size :", len(random_indices))
    print("Overlap     :", overlap)
else:
    print(
        "WARNING: `_candidate_index` metadata not present in all LESS records. "
        "Training can still proceed from the `messages` field."
    )

assert all("messages" in r for r in less_records), (
    "LESS selection must preserve the original `messages` field."
)

print("Dataset integrity checks passed.")


LESS candidate-index metadata found.
LESS size   : 450
Random size : 450
Overlap     : 21
Dataset integrity checks passed.


In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer:", MODEL_NAME)
print("Vocab    :", len(tokenizer))

def tokenize_instruction_example(example):
    encoded = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )

    # Mirror Phase 1's robust handling of tokenizer outputs.
    if hasattr(encoded, "encodings") and encoded.encodings:
        input_ids = encoded.encodings[0].ids
    elif hasattr(encoded, "input_ids"):
        input_ids = encoded["input_ids"]
        if input_ids and isinstance(input_ids[0], list):
            input_ids = input_ids[0]
    else:
        input_ids = encoded

    input_ids = list(input_ids)[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": input_ids.copy(),
        "attention_mask": [1] * len(input_ids),
    }

def prepare_training_dataset(ds, name):
    tokenized = ds.map(
        tokenize_instruction_example,
        remove_columns=ds.column_names,
        desc=f"Tokenizing {name}",
    )
    return tokenized

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("Tokenizer preparation complete.")


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B
Vocab    : 151665
Tokenizer preparation complete.


In [11]:
def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
        trust_remote_code=True,
    )

    model.config.use_cache = False
    model.to(DEVICE)

    return model


def attach_lora(model):
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
    )

    model = get_peft_model(model, lora_config)
    return model


def prepare_training_model():
    model = load_base_model()
    model = attach_lora(model)

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    model.config.use_cache = False
    model.enable_input_require_grads()

    try:
        model.base_model.model.config._attn_implementation = "eager"
        transformer = model.base_model.model.model
        lm_head = model.base_model.model.lm_head
    except AttributeError:
        transformer = model.base_model.model.model
        lm_head = model.base_model.model.lm_head

    model.train()

    fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
        reduction="mean",
        ignore_index=-100,
    )

    return model, transformer, lm_head, fused_loss_fn


def cleanup_model(model):
    try:
        model.zero_grad(set_to_none=True)
    except Exception:
        pass

    del model
    gc.collect()
    torch.cuda.empty_cache()


def get_lora_parameters(model):
    return [p for p in model.parameters() if p.requires_grad]


def gpu_memory(tag):
    allocated = torch.cuda.memory_allocated(DEVICE) / 1024**3
    reserved = torch.cuda.memory_reserved(DEVICE) / 1024**3
    peak = torch.cuda.max_memory_allocated(DEVICE) / 1024**3

    print(
        f"[{tag}] allocated={allocated:.2f} GiB | "
        f"reserved={reserved:.2f} GiB | peak={peak:.2f} GiB"
    )


In [12]:
def train_experiment(experiment_name, raw_dataset):
    print()
    print("=" * 80)
    print(f"TRAINING: {experiment_name}")
    print("=" * 80)

    experiment_root = EXPERIMENT_ROOT / experiment_name
    checkpoint_root = experiment_root / "checkpoints"
    checkpoint_root.mkdir(parents=True, exist_ok=True)

    tokenized = prepare_training_dataset(
        raw_dataset,
        experiment_name,
    )

    train_loader = DataLoader(
        tokenized,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        collate_fn=data_collator,
    )

    num_micro_batches = len(train_loader)
    steps_per_epoch = math.ceil(
        num_micro_batches / GRAD_ACCUM_STEPS
    )
    total_training_steps = steps_per_epoch * NUM_EPOCHS
    num_warmup_steps = int(
        WARMUP_RATIO * total_training_steps
    )

    print(f"Examples                 : {len(raw_dataset):,}")
    print(f"Micro-batches / epoch    : {num_micro_batches:,}")
    print(f"Optimizer steps / epoch : {steps_per_epoch:,}")
    print(f"Total optimizer steps   : {total_training_steps:,}")
    print(f"Warmup steps            : {num_warmup_steps:,}")

    model, transformer, lm_head, fused_loss_fn = prepare_training_model()

    model.print_trainable_parameters()
    gpu_memory("after model load")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps,
    )

    global_step = 0
    training_log = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        running_loss = 0.0
        epoch_lrs = []

        optimizer.zero_grad(set_to_none=True)

        progress = tqdm(
            train_loader,
            desc=f"{experiment_name} | epoch {epoch}/{NUM_EPOCHS}",
        )

        for step, batch in enumerate(progress, start=1):

            batch = {
                k: v.to(
                    DEVICE,
                    non_blocking=True,
                )
                for k, v in batch.items()
            }

            labels = batch["labels"].reshape(-1)

            with torch.autocast(
                device_type="cuda",
                dtype=DTYPE,
            ):
                outputs = transformer(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    use_cache=False,
                    return_dict=True,
                )

                hidden_states = outputs.last_hidden_state.reshape(
                    -1,
                    outputs.last_hidden_state.shape[-1],
                )

                loss = fused_loss_fn(
                    lm_head.weight,
                    hidden_states,
                    labels,
                )

                loss_for_backward = (
                    loss / GRAD_ACCUM_STEPS
                )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at {experiment_name}, "
                    f"epoch={epoch}, step={step}"
                )

            loss_value = float(loss.detach().item())
            loss_for_backward.backward()
            running_loss += loss_value

            should_step = (
                step % GRAD_ACCUM_STEPS == 0
                or step == len(train_loader)
            )

            if should_step:
                torch.nn.utils.clip_grad_norm_(
                    get_lora_parameters(model),
                    max_norm=MAX_GRAD_NORM,
                )

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                current_lr = scheduler.get_last_lr()[0]
                epoch_lrs.append(current_lr)

                progress.set_postfix(
                    loss=f"{loss_value:.4f}",
                    lr=f"{current_lr:.3e}",
                    step=global_step,
                    gpu=f"{torch.cuda.memory_allocated(DEVICE)/1024**3:.2f}G",
                )

            del batch, labels, outputs
            del hidden_states, loss, loss_for_backward

            gc.collect()

        avg_loss = running_loss / len(train_loader)
        avg_lr = (
            float(np.mean(epoch_lrs))
            if epoch_lrs
            else 0.0
        )

        print(
            f"\n{experiment_name} | epoch {epoch}: "
            f"loss={avg_loss:.5f}, "
            f"avg_lr={avg_lr:.6e}, "
            f"global_step={global_step}"
        )
        gpu_memory(f"{experiment_name} epoch {epoch}")

        checkpoint_dir = checkpoint_root / f"epoch_{epoch}"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        model.save_pretrained(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)

        training_state = {
            "experiment": experiment_name,
            "epoch": epoch,
            "global_step": global_step,
            "avg_loss": avg_loss,
            "avg_lr": avg_lr,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
            "micro_batch_size": MICRO_BATCH_SIZE,
            "effective_batch_size": EFFECTIVE_BATCH_SIZE,
            "max_grad_norm": MAX_GRAD_NORM,
            "warmup_ratio": WARMUP_RATIO,
            "num_epochs": NUM_EPOCHS,
            "seed": SEED,
        }

        with open(
            checkpoint_dir / "training_meta.json",
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(training_state, f, indent=2)

        torch.save(
            {
                "epoch": epoch,
                "global_step": global_step,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "loss": avg_loss,
            },
            checkpoint_dir / "training_state.pt",
        )

        training_log.append(training_state)

        print("Checkpoint saved:", checkpoint_dir)

        gc.collect()
        torch.cuda.empty_cache()

    with open(
        experiment_root / "training_log.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(training_log, f, indent=2)

    cleanup_model(model)

    return experiment_root


In [13]:
EXPERIMENT_DATA = {}

if RUN_RANDOM_450:
    EXPERIMENT_DATA["random450"] = random_dataset

if RUN_LESS_450:
    EXPERIMENT_DATA["less450"] = less_dataset

print("Experiments to train:")
for name, ds in EXPERIMENT_DATA.items():
    print(f"  {name:10s}: {len(ds):,} examples")

print("\nEach condition starts from a fresh Qwen2.5-1.5B base model.")
print("There is NO 9,000-example retraining in this notebook.")


Experiments to train:
  random450 : 450 examples
  less450   : 450 examples

Each condition starts from a fresh Qwen2.5-1.5B base model.
There is NO 9,000-example retraining in this notebook.


In [14]:
TRAINED_EXPERIMENT_ROOTS = {}

for experiment_name, dataset in EXPERIMENT_DATA.items():
    root = train_experiment(
        experiment_name,
        dataset,
    )
    TRAINED_EXPERIMENT_ROOTS[experiment_name] = root

print()
print("=" * 80)
print("RANDOM-450 + LESS-450 TRAINING COMPLETE")
print("=" * 80)

for name, root in TRAINED_EXPERIMENT_ROOTS.items():
    print(name, "->", root)

print("\nEach experiment should contain:")
print("  checkpoints/epoch_1")
print("  checkpoints/epoch_2")
print("  checkpoints/epoch_3")
print("  checkpoints/epoch_4")



TRAINING: random450


Tokenizing random450:   0%|          | 0/450 [00:00<?, ? examples/s]

Examples                 : 450
Micro-batches / epoch    : 450
Optimizer steps / epoch : 4
Total optimizer steps   : 16
Warmup steps            : 0


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...


less450 | epoch 2: loss=9.53156, avg_lr=1.283336e-05, global_step=8
[less450 epoch 2] allocated=3.28 GiB | reserved=3.82 GiB | peak=4.41 GiB
Checkpoint saved: less_phase4_training/experiments/less450/checkpoints/epoch_2


less450 | epoch 3/4:   0%|          | 0/450 [00:00<?, ?it/s]


less450 | epoch 3: loss=8.33488, avg_lr=5.398873e-06, global_step=12
[less450 epoch 3] allocated=3.28 GiB | reserved=3.83 GiB | peak=4.41 GiB
Checkpoint saved: less_phase4_training/experiments/less450/checkpoints/epoch_3


less450 | epoch 4/4:   0%|          | 0/450 [00:00<?, ?it/s]


less450 | epoch 4: loss=7.85948, avg_lr=6.596639e-07, global_step=16
[less450 epoch 4] allocated=3.28 GiB | reserved=3.83 GiB | peak=4.41 GiB
Checkpoint saved: less_phase4_training/experiments/less450/checkpoints/epoch_4

RANDOM-450 + LESS-450 TRAINING COMPLETE
random450 -> less_phase4_training/experiments/random450
less450 -> less_phase4_training/experiments/less450

Each experiment should contain:
  checkpoints/epoch_1
  checkpoints/epoch_2
  checkpoints/epoch_3
  checkpoints/epoch_4


## Output

After execution:

```text
less_phase4_training/
└── experiments/
    ├── random450/
    │   └── checkpoints/
    │       ├── epoch_1/
    │       ├── epoch_2/
    │       ├── epoch_3/
    │       └── epoch_4/
    └── less450/
        └── checkpoints/
            ├── epoch_1/
            ├── epoch_2/
            ├── epoch_3/
            └── epoch_4/
```

The next notebook consumes these checkpoints and evaluates:
- Base Qwen2.5-1.5B
- existing Phase-1 Full-9000 checkpoints
- Random-450 epochs 1–4
- LESS-450 epochs 1–4
